# Human in the Loop 
This is a concept where we can include the human intervention to the agent 
like giving inputs
it works like first we will send prompt -> then the llm analyzes the requirement toolcall -> if that tool requires human input, it interrupts the flow -> then takes the input from Command(resume={}) -> then gives the output

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')
os.environ['TAVILY_API_KEY']=os.getenv('TAVILY_API_KEY')

In [11]:
from langchain.chat_models import init_chat_model

llm=init_chat_model(
    model="groq:qwen/qwen3-32b"
)

In [12]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

class State(TypedDict):
    messages:Annotated[list,add_messages]

graph_builder=StateGraph(State)

In [13]:
from langchain_tavily import TavilySearch

tool1=TavilySearch(max_results=2)

## Interrupt and Checkpointer
Without checkpointer, there is no use of interrupt. Since without checkpointer there will be no storage of the conversation

## Working of Human In the Loop Middleware

In [14]:
from langgraph.types import interrupt,Command
from langchain.tools import tool

@tool
def human_assitance(query:str)->str:
    """
    I need you to provide assistance to build an ai agent
    Use this tool whenever additional information,
    clarification, approval, or feedback is required
    from the human user before continuing.
    """
    print("waiting for human response")
    human_response=interrupt({"query":query})
    print(human_response)
    print(type(human_response))
    return human_response["data"]

tools=[tool1,human_assitance]

llm_with_tools=llm.bind_tools(tools)

In [15]:
def chatbot(state:State):
    return {"messages":[llm_with_tools.invoke(state["messages"])]}

graph_builder.add_node('chatbot',chatbot)
graph_builder.add_node('tools',ToolNode(tools))
graph_builder.add_edge(START,'chatbot')
graph_builder.add_conditional_edges(
    'chatbot', tools_condition
)
graph_builder.add_edge('tools','chatbot')
graph_builder.add_edge('chatbot',END)

graph=graph_builder.compile(MemorySaver())

### Stream Modes
There are different stream modes:
- stream mode: this is a synchronous mode like requests.get()
- astream mode:  this is a asynchronous mode like await requests.get()

in this different modes also there are 2 types:
- values: values keeps appending to the list, like if response is a list, the values mode will keep appending the response word by word given by llm
- updates: updates mode only stores the last updated piece of text in the response

In [19]:
from pprint import pprint

user_input="I need you to provide assistance to build an ai agent"
config={'configurable':{'thread_id':'t1'}}

events=graph.stream({'messages':user_input},config=config,stream_mode='values')
for event in events:
    if "messages" in event:
        pprint(event)
        # print(event["messages"][-1].tool_calls)


{'messages': [HumanMessage(content='I need you to provide assistance to build an ai agent', additional_kwargs={}, response_metadata={}, id='6a4295f3-b677-4f60-9fe9-03483fc11106'),
              AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants help building an AI agent. Let me start by understanding what they need. They mentioned using the human_assitance tool when more info is needed. Since they're asking for assistance, I should break down the process.\n\nFirst, I need to ask them about the specific tasks the AI agent should handle. Are they looking for something like customer service, data analysis, or maybe a personal assistant? The tools available include a search function, so maybe they need the agent to perform searches or gather information.\n\nNext, I should check if they have any preferences for the agent's capabilities. Do they want it to use the Tavily search tool for real-time data? Or perhaps they need image search or domain-specific info

## Giving input after interrupt

In [20]:
human_response="I am an agent building expert, and i would like you to look into the langgraph to build the ai agent for myslef"

human_command=Command(resume={"data":human_response})

events=graph.stream(human_command,config=config, stream_mode="values")
for event in events:
    if "messages" in event:
        pprint(event)



{'messages': [HumanMessage(content='I need you to provide assistance to build an ai agent', additional_kwargs={}, response_metadata={}, id='6a4295f3-b677-4f60-9fe9-03483fc11106'),
              AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants help building an AI agent. Let me start by understanding what they need. They mentioned using the human_assitance tool when more info is needed. Since they're asking for assistance, I should break down the process.\n\nFirst, I need to ask them about the specific tasks the AI agent should handle. Are they looking for something like customer service, data analysis, or maybe a personal assistant? The tools available include a search function, so maybe they need the agent to perform searches or gather information.\n\nNext, I should check if they have any preferences for the agent's capabilities. Do they want it to use the Tavily search tool for real-time data? Or perhaps they need image search or domain-specific info